# 연습 경기장 — A/B구역 ArUco 4개 수집 + 정지 후 YOLO 순찰 + 초록 복귀

A구역 → 순찰 P1~P4 → B구역 → 순찰 P5~P6 → 초록 검출 정지 순으로 주행합니다.
기존 "회전하며 인식(우270)" 대신, 순찰 지점마다 완전히 정지한 뒤 YOLO를 한 번 추론하고
그 결과를 기록한 다음 좌90/180을 수행합니다(멀티스레드 불필요).

| 상태 | 도달 조건 | 행동 |
|---|---|---|
| 1 | A구역 진입 초록 검출 | 정지 → 좌90 |
| 2 | A구역 실물 마커 4장 확인 (중복 ID 허용) | 정지 → 180 |
| 3 | A구역 복귀 중 초록 검출 | 정지 → 좌90 |
| 4 | 순찰 P1 거리 도달 | 정지 → YOLO 인식 → 좌90 |
| 5 | 순찰 P2 (P1 직후) | 정지 → YOLO 인식 → 180 |
| 6 | 순찰 P3 거리 도달 | 정지 → YOLO 인식 → 좌90 |
| 7 | 순찰 P4 (P3 직후) | 정지 → YOLO 인식 → 180 |
| 8 | B구역 진입 초록 검출 | 정지 → 좌90 |
| 9 | B구역 실물 마커 4장 확인 (중복 ID 허용) | 정지 (회전 없음) |
| 10 | B구역 복귀 중 초록 검출 | 정지 → 좌90 |
| 11 | 순찰 P5 거리 도달 | 정지 → YOLO 인식 → 좌90 |
| 12 | 순찰 P6 (P5 직후) | 정지 → YOLO 인식 → 180 |
| 13 | 최종 복귀 중 초록 검출 | 정지 (완주) |

## ArUco 실물 4장 추적

- ID 종류가 아니라 화면의 **서로 다른 위치에 있는 실물 마커 4장**을 셉니다.
  같은 ID도 허용하므로 `[7, 7, 19, 37]`이 정상적인 완료 리스트입니다.
- 검출 중심·크기·ID를 비교해 프레임 사이의 같은 마커를 연결합니다.
  `ARUCO_CONFIRM_FRAMES=2`번 관측되면 해당 장을 확정합니다.
  연속 프레임일 필요는 없으며 잠깐 놓치면 관측 횟수와 확정 상태를 유지합니다.
- `ARUCO_TRACK_TTL_S=0.8`초 이내 누락만 유지합니다. 하나라도 더 오래 놓치면
  유령 중복 집계를 피하려고 **해당 구역 묶음 전체를 다시 확인**합니다.
  마커 4장이 함께 시야에 들어오는 배치용이며, 장기 가림/교차를 해결하는 추적기는 아닙니다.
- 연결 허용 이동은 `ARUCO_TRACK_MAX_SHIFT=0.8` × 마커 변 길이(최소 4px),
  크기 비율은 `ARUCO_TRACK_MAX_SCALE=1.6`입니다.
  두 후보에 걸리거나 놓친 동일 ID가 먼 위치에 나타나면 새 장으로 바로 추가하지 않습니다.
  연결이 불확실하면 기존 위치 트랙은 유지하되 네 장의 확인 횟수를 다시 받습니다.
- 1~3장에서는 기존 접근 출력으로 라인을 따라 전진합니다. 최근 추적 중인 네 장이
  모두 확정되고 현재 연결에 모호함이 없으면 정지 → 저장 → IMU 180도 회전합니다.
  불확실하면 완료 보류하며 기존 거리·시간 상한에서 안전 정지합니다.
- 이벤트에 `IDs=[...] (n/4장)`과 `T1:id7=2*`처럼 추적번호·ID·관측 횟수를 표시합니다.
  별표는 확정입니다. 동일 ID 두 장은 T1/T2 등 서로 다른 번호로 표시됩니다.
- `DRIVE['aruco_zone_ids']`와 JSON의 `zone_ids`는 중복 ID를 보존합니다.
  `zone_tracks`는 위치/크기/관측 횟수, `zone_complete`는 실제 완료 판정입니다.
  기존 `seen_ids`, JSON의 `all_ids/new_ids/sum`은 ID 종류 통계일 뿐 장수 판정이 아닙니다.
- 상단은 하단과 별도로 추적합니다. 회전 후 유예 `ARUCO_COOLDOWN_S=0.7`은 유지합니다.

## 초록 복귀

진입거리=복귀거리 조건을 제거했습니다. 복귀 중 엔코더는 안전 상한 확인에만 사용합니다.
첨부 `linetracing_test.ipynb` 맨 위 테스트의 초록 검출을 참고합니다:

- HSV: `GREEN_HSV_LOWER=[42,72,90]`, `GREEN_HSV_UPPER=[72,199,207]`.
- 검은 ROI와 같은 폭, 높이 1/3인 띠를 중앙에서 아래로 `GREEN_OFFSET_Y=20`px 이동합니다.
  현재 320×180 기본 ROI에서는 x=76~203, y=140~169입니다.
- 3×3 OPEN/CLOSE 후 최대 윤곽선 면적 `MIN_GREEN_AREA=60` 이상이면 검출합니다.
  **주행 화면에 그린 초록 윤곽선이 아니라 원본 카메라 픽셀을 검사**합니다.
- 복귀 출력 `GREEN_RETURN_SPEED=60`. 첫 유효 초록부터 정지하고,
  `GREEN_CONFIRM_FRAMES=2` 연속 확인하면 좌90합니다. 확인 중 초록이 사라지면 다시 탐색합니다.
- `GREEN_IGNORE_INITIAL_VISIBLE=True`: 180도 직후 이미 보이는 초록은 잔류로 간주합니다.
  한 번 안 보인 뒤 새로 보이는 초록부터 받습니다. 이벤트의 `WAIT_CLEAR`는 이탈 대기입니다.
  **직후 보이는 초록 자체가 원하는 이탈 마커인 배치라면** 이 값을 False로 조정해야 합니다.
- 초록 설정은 이 노트북의 `GREEN_HSV_*`를 사용하며 공용 calib 저장값이 자동 적용되지는 않습니다.

## 안전 상한과 현장 확인

다음은 **임시 안전 상한**이며 원하는 주행거리나 회전 목표가 아닙니다. 현장에서 반드시 확인하세요.

- `ARUCO_MAX_APPROACH_M={'하단':0.80,'상단':0.80}`, `ARUCO_SCAN_TIMEOUT_S=15.0`.
- `GREEN_RETURN_MAX_M={'하단':1.20,'상단':1.20}`, `GREEN_RETURN_TIMEOUT_S=15.0`.
- 한도에 도달하면 미션을 성공 처리하거나 임의 회전하지 않고 안전 정지합니다.
- 선 소실 시 정지는 유지합니다. 정지해도 카메라 인식은 계속하므로 보이는 ID는 수집할 수 있습니다.
- 한도는 루프와 영상 수신/분석 전후에 검사합니다. 카메라 호출이 영구 정지하는 상황을
  독립적으로 제어하는 하드웨어 watchdog은 아니므로 물리 정지 수단도 준비해야 합니다.
- 고정 거리 6개 `DISTANCE_M`는 그대로입니다. 복귀 좌90 위치가 초록 기준으로 바뀌므로
  `T02_LOWER_EXIT_TO_PATROL_1`, `T05_UPPER_EXIT_TO_PATROL_3`는 새 위치에서 다시 확인하세요.

## 적용

현재 저장된 검은선 설정·P/IMU 조향·IMU 회전은 변경하지 않았습니다.
사용자가 설정한 검정 상한 120, 거리값, DISTANCES_CONFIGURED 설정은 보존했습니다.
**적용 전에 실물의 검정 설정과 거리값이 동일한지 확인**하세요.
초록 HSV도 조명에 따라 현장 조절해야 합니다.

기존 주행 정지 → 정리 셀 → 커널 재시작 후, 고정 거리 6개와 안전 상한/색상 값을 확인하고
`DISTANCES_CONFIGURED=True`로 설정합니다.
파라미터 → 초기화 → 함수 정의 → 주행 UI 셀 순서로 실행합니다.
모터를 움직이기 전에 초기화 출력의 검정·초록 HSV가 의도한 값인지 확인하세요.

기존 03 및 Webots 본선 월드는 변경하지 않았습니다. 06은 독립 사본이며 이 맵의 Webots 연동은 포함하지 않습니다.
실제 물체 분류기는 `OBJECT_CLASSIFIER`에 연결해야 합니다(미연결 시 UNKNOWN).
오프라인 검증과 별개로 실물의 인식 위치·정지 관성·미끄러짐은 현장에서 확인해야 합니다.


In [1]:
# ===== 테스트 경기장 파라미터 + 거리 기반 10상태 FSM =====

# --- 거리 측정 ---
DISTANCES_CONFIGURED = True # 아래 6개 실측값을 넣은 뒤 True

# 전부 임시값. 직전 행동 완료 위치부터 다음 거리 지점까지의 직선 주행거리(m).
DISTANCE_M = {
    # 전부 TODO 임시값 — 26단계 연습경기장 실측 후 반드시 교체할 것.
    "L01_RETURN_A_TO_P1": 0.60,   # A구역 복귀 좌90 완료 → 순찰 P1
    "L02_P1_TO_P2": 0.05,         # P1 좌90 완료 직후 바로 인식(사실상 거리 없음)
    "L03_P2_TO_P3": 0.60,         # P2 180 완료 → 순찰 P3
    "L04_P3_TO_P4": 0.05,         # P3 좌90 완료 직후 바로 인식
    "L05_RETURN_B_TO_P5": 0.60,   # B구역 복귀 좌90 완료 → 순찰 P5
    "L06_P5_TO_P6": 0.05,         # P5 좌90 완료 직후 바로 인식
}

# 하단/상단은 ArUco 4개 수집 후 180도, 복귀는 초록색 검출로 좌90.
# 진입거리=복귀거리 조건은 사용하지 않음.
# test.ipynb와 같은 엔코더 환산. None이면 calib 값 또는 반지름/틱수로 계산.
METERS_PER_TICK_OVERRIDE = None
DISTANCE_SCALE = 1.0
DISTANCE_STOP_EARLY_M = 0.03
DISTANCE_SLOWDOWN_M = 0.15
DISTANCE_SLOW_FACTOR = 0.60
DISTANCE_SEGMENT_TIMEOUT_S = 30.0

# --- 검은선 추종: 첨부 linetracing_test.ipynb 맨 위 '아래는 테스트' 기준 ---
LINE_BLACK_LOWER = [0, 0, 0]
LINE_BLACK_UPPER = [179, 255, 120]
LINE_ROI_TOP_FRAC = 0.50
LINE_ROI_WIDTH_RATIO = 0.40
LINE_ROI_OFFSET_X = -20           # 320x180 기준 픽셀 단위
LINE_MIN_CONTOUR_AREA = 80        # 3x3 OPEN/CLOSE 후 최소 윤곽선 면적

# --- 주행 보정: linetracingtest.ipynb(밑줄 없음)의 P + 약한 IMU ---
LINE_CONTROL_MODE = 'P_IMU'      # 'P_IMU': 이번 보정 / 'PD_TEST': 이전 PD 계산
LINE_P_KP = 0.18
LINE_P_MAX_STEER = 12.0          # 좌/우 기본 출력에 더하고 빼는 보정량 상한
LINE_P_REFERENCE_WIDTH = 640     # 원본 640px → 현재 영상 폭으로 픽셀 오차 환산
LINE_USE_IMU_CORRECTION = True   # 선 P 보정 + 직진 IMU 보정. 제자리 회전과 별개

# 이전 PD_TEST 전용. P_IMU에서는 D항/중앙 무보정/조향 시 감속을 사용하지 않음.
LINE_KP = 0.30
LINE_KD = 0.12                    # 픽셀 오차의 프레임 간 차이 (초 단위 미분 아님)
LINE_DEAD_ZONE_PX = 18           # CENTER 표시에도 사용. P_IMU는 이 안에서도 보정
LINE_MID_ERROR_PX = 40
LINE_MAX_SPEED = 120              # 조향할 때만 적용. 중앙 직진은 BASE 출력 사용
LINE_MID_SPEED_FACTOR = 0.90
LINE_LARGE_SPEED_FACTOR = 0.70
LINE_LARGE_KP_FACTOR = 1.35

# 두 보정 모드 공통 화면 갱신 설정
LINE_DISPLAY_EVERY = 2

# --- 일반 출력 / 선택적 직진 IMU 보정 ---
YAW_KP = 0.01
YAW_MAX_CORRECTION = 0.5
YAW_DEADBAND_DEG = 1.0           # 밑줄 없는 원본의 yaw_deadband
BASE_SPEED_OVERRIDE = 140
LOOP_DT = 0.03
YAW_SIGN_OVERRIDE = -1
ENABLE_ARUCO = True

# --- ArUco: ID를 미리 지정하지 않고, 인식된 마커로 180도 회전 ---
# 아래 ID는 구역별 예시일 뿐, 실물 인식 조건에는 사용하지 않는다.
MARKER_IDS = {"A": 0, "B": 1}
ARUCO_REQUIRED_COUNT = 4         # 서로 다른 실물 마커 수. 같은 ID도 별도 위치면 허용
ARUCO_CONFIRM_FRAMES = 2         # 추적 중 관측 횟수. 잠깐 누락되어도 횟수 유지
ARUCO_TRACK_TTL_S = 0.8          # 마지막 관측 후 유지 시간. 초과 시 해당 구역 재확인
ARUCO_TRACK_MAX_SHIFT = 0.8      # 연결 허용 중심 이동 / 마커 한 변 길이
ARUCO_TRACK_MAX_SCALE = 1.6      # 연결 허용 크기 비율(큰 변 / 작은 변)
ARUCO_TRACK_DUPLICATE_RATIO = 0.35  # 같은 프레임 같은 위치 중복 검출 제거
ARUCO_APPROACH_SPEED = 60.0  # 하단/상단 접근 기본 출력; BASE보다 높이지 않음 (m/s 아님)
ARUCO_SCAN_TIMEOUT_S = 15.0      # 진입 회전 완료부터 수집 제한시간
ARUCO_MAX_APPROACH_M = {'A': 0.80, 'B': 0.80}  # 안전 상한 임시값: 현장 확인 필수
ARUCO_MIN_SIDE_PX = 0
ARUCO_COOLDOWN_S = 0.7
ARUCO_OLED_LOG = True
ARUCO_LOG_PATH = "aruco_seen_test_course.json"

# --- 복귀 전용 초록 검출: 첨부 linetracing_test.ipynb 맨 위 테스트 ---
GREEN_HSV_LOWER = [42, 72, 90]
GREEN_HSV_UPPER = [72, 199, 207]
GREEN_OFFSET_Y = 20              # 검은 ROI 중앙의 높이 1/3 띠를 아래로 이동(px)
MIN_GREEN_AREA = 60              # 3x3 OPEN/CLOSE 후 최대 윤곽선 면적
GREEN_CONFIRM_FRAMES = 2         # 첫 유효 검출부터 정지하며 연속 확인
GREEN_IGNORE_INITIAL_VISIBLE = True  # 회전 직후 잔류 초록은 벗어난 후 재검출
GREEN_RETURN_SPEED = 60.0
GREEN_RETURN_TIMEOUT_S = 15.0
GREEN_RETURN_MAX_M = {'A': 1.20, 'B': 1.20}  # 초록 누락 시 멈출 안전 상한 임시값

# --- 물체 인식 ---
OBJECT_CONFIRM_FRAMES = 3
OBJECT_MIN_CONFIDENCE = 0.60
OBJECT_LABELS = {"CONE", "HELMET"}
OBJECT_SAMPLE_INTERVAL_S = 0.03  # 회전 중 추론 요청 간격

# --- 제자리 회전 ---
TURN_POWER = 45
TURN_STOP_EARLY_DEG = 6.0
TURN_TIMEOUT_S = 5.0
TURN_SETTLE_S = 0.3
TURN_TOL_DEG = 5.0
TURN_PRESTOP_S = 0.6
HEADING_RESYNC = 0.0             # 직선 구간 중 IMU 기준을 따라 움직이지 않음

# --- 안전 ---
MAX_RUN_TIME = 300.0
LOW_BATT_V = 10.5

# DISTANCE_YOLO = 거리까지 직진 후 완전히 정지, 회전 없이 YOLO 한 번만 추론해 기록한 다음
# action(좌90/180)을 수행한다. 기존 DISTANCE_OBJECT(회전 중 인식, 스레드 필요)는 이 코스에서 쓰지 않음.
# 동선: A구역 진입→마커4개→180→복귀 → 순찰 P1~P4 → B구역 진입→마커4개(180 없음)→복귀 → 순찰 P5~P6 → 초록 정지
MISSION_PLAN = [
    {"event": "GREEN_RETURN", "return_target": "A", "action": "LEFT_90", "name": "1. A구역 진입 초록 → 좌90"},
    {"event": "ARUCO", "target": "A", "action": "TURN_180", "name": "2. A구역 마커 4개 수집 → 180"},
    {"event": "GREEN_RETURN", "return_target": "A", "action": "LEFT_90", "name": "3. A구역 복귀 초록 → 좌90"},
    {"event": "DISTANCE_YOLO", "zone": "P1", "distance_m": DISTANCE_M["L01_RETURN_A_TO_P1"], "action": "LEFT_90", "name": "4. 순찰 P1 정지 → YOLO → 좌90"},
    {"event": "DISTANCE_YOLO", "zone": "P2", "distance_m": DISTANCE_M["L02_P1_TO_P2"], "action": "TURN_180", "name": "5. 순찰 P2 정지 → YOLO → 180"},
    {"event": "DISTANCE_YOLO", "zone": "P3", "distance_m": DISTANCE_M["L03_P2_TO_P3"], "action": "LEFT_90", "name": "6. 순찰 P3 정지 → YOLO → 좌90"},
    {"event": "DISTANCE_YOLO", "zone": "P4", "distance_m": DISTANCE_M["L04_P3_TO_P4"], "action": "TURN_180", "name": "7. 순찰 P4 정지 → YOLO → 180"},
    {"event": "GREEN_RETURN", "return_target": "B", "action": "LEFT_90", "name": "8. B구역 진입 초록 → 좌90"},
    {"event": "ARUCO", "target": "B", "action": "NONE", "name": "9. B구역 마커 4개 수집 (회전 없음)"},
    {"event": "GREEN_RETURN", "return_target": "B", "action": "LEFT_90", "name": "10. B구역 복귀 초록 → 좌90"},
    {"event": "DISTANCE_YOLO", "zone": "P5", "distance_m": DISTANCE_M["L05_RETURN_B_TO_P5"], "action": "LEFT_90", "name": "11. 순찰 P5 정지 → YOLO → 좌90"},
    {"event": "DISTANCE_YOLO", "zone": "P6", "distance_m": DISTANCE_M["L06_P5_TO_P6"], "action": "TURN_180", "name": "12. 순찰 P6 정지 → YOLO → 180"},
    {"event": "GREEN_RETURN", "return_target": "B", "action": "FINISH", "name": "13. 최종 복귀 초록 → 정지"},
]

assert len(MISSION_PLAN) == 13
assert len(DISTANCE_M) == 6 and all(0 < float(v) < float('inf') for v in DISTANCE_M.values())
assert [s['return_target'] for s in MISSION_PLAN if 'return_target' in s] == ['A', 'A', 'B', 'B', 'B']
assert [s.get("zone") for s in MISSION_PLAN if s["event"] == "DISTANCE_YOLO"] == ["P1", "P2", "P3", "P4", "P5", "P6"]
assert MISSION_PLAN[-1]["action"] == "FINISH"
assert 0 < DISTANCE_SLOW_FACTOR <= 1.0
assert set(MARKER_IDS) == {"A", "B"}
assert all(0 <= int(v) < 50 for v in MARKER_IDS.values())  # 예시 ID 사전 범위
assert ARUCO_CONFIRM_FRAMES >= 1 and ARUCO_MIN_SIDE_PX >= 0
assert 0 < ARUCO_APPROACH_SPEED <= 180
assert 0 < ARUCO_TRACK_TTL_S < float('inf')
assert 0 < ARUCO_TRACK_MAX_SHIFT < float('inf')
assert 1 < ARUCO_TRACK_MAX_SCALE < float('inf')
assert 0 < ARUCO_TRACK_DUPLICATE_RATIO < ARUCO_TRACK_MAX_SHIFT
assert isinstance(ARUCO_REQUIRED_COUNT, int) and 1 <= ARUCO_REQUIRED_COUNT <= 50
assert isinstance(ARUCO_CONFIRM_FRAMES, int) and ARUCO_CONFIRM_FRAMES >= 1
assert 0 < ARUCO_SCAN_TIMEOUT_S < float('inf') and 0 < GREEN_RETURN_TIMEOUT_S < float('inf')
assert 0 < GREEN_RETURN_SPEED <= 180
assert isinstance(GREEN_CONFIRM_FRAMES, int) and GREEN_CONFIRM_FRAMES >= 1
assert isinstance(GREEN_IGNORE_INITIAL_VISIBLE, bool) and MIN_GREEN_AREA > 0
assert len(GREEN_HSV_LOWER) == len(GREEN_HSV_UPPER) == 3
assert all(0 <= lo <= hi <= cap for lo, hi, cap in zip(GREEN_HSV_LOWER, GREEN_HSV_UPPER, [179,255,255]))
assert set(ARUCO_MAX_APPROACH_M) == set(GREEN_RETURN_MAX_M) == {'A', 'B'}
assert all(0 < float(v) < float('inf') for limits in (ARUCO_MAX_APPROACH_M, GREEN_RETURN_MAX_M) for v in limits.values())
assert LINE_CONTROL_MODE in ('P_IMU', 'PD_TEST')
assert 0 <= LINE_P_KP < float('inf') and 0 <= LINE_P_MAX_STEER <= 180
assert 0 < LINE_P_REFERENCE_WIDTH < float('inf')
assert 0 <= YAW_KP < float('inf') and 0 <= YAW_MAX_CORRECTION <= 180
assert 0 <= YAW_DEADBAND_DEG <= 180
assert len(LINE_BLACK_LOWER) == len(LINE_BLACK_UPPER) == 3
assert all(0 <= lo <= hi <= cap for lo, hi, cap in zip(LINE_BLACK_LOWER, LINE_BLACK_UPPER, [179, 255, 255]))
assert 0 <= LINE_ROI_TOP_FRAC < 1 and 0 < LINE_ROI_WIDTH_RATIO <= 1
assert LINE_MIN_CONTOUR_AREA > 0 and LINE_KP >= 0 and LINE_KD >= 0
assert 0 <= LINE_DEAD_ZONE_PX < LINE_MID_ERROR_PX and 0 < LINE_MAX_SPEED <= 180
assert 0 < LINE_LARGE_SPEED_FACTOR <= LINE_MID_SPEED_FACTOR <= 1 and LINE_LARGE_KP_FACTOR > 0
assert isinstance(LINE_DISPLAY_EVERY, int) and LINE_DISPLAY_EVERY >= 1
print("연습경기장 13상태 FSM | 구간거리 6개 + 초록 복귀/진입 5개 | A-P1~P4-B-P5~P6-초록정지 | 설정완료=%s" % DISTANCES_CONFIGURED)


테스트 경기장 10상태 FSM | 입력 6개 + 초록 복귀 2개 | 하단-#1-#2-상단-#3-정지 | 설정완료=True


In [2]:
# ===== 초기화 + tiki_calib.json 로드 (세션당 1번) =====
import cv2
import numpy as np
import time
import math
import json
import copy
import os
from datetime import datetime

import threading
import ipywidgets as widgets
from IPython.display import display

from tiki.mini import TikiMini
from picamera2 import Picamera2

CALIB_PATH = 'tiki_calib.json'

_stop_events = []


def make_stop_event():
    e = threading.Event()
    _stop_events.append(e)
    return e

DEFAULT_CALIB = {
    'camera': {'size': [320, 180], 'format': 'BGR888', 'rgb2bgr': True,
               'controls': {'AwbEnable': False, 'ColourGains': [0.0, 0.0],
                            'AeEnable': False, 'ExposureTime': 0, 'AnalogueGain': 0.0},
               'locked': False},
    'colors': {'black_line': {'lower': [0, 0, 0], 'upper': [179, 255, 67]},
               'green_marker': {'lower': [40, 80, 60], 'upper': [85, 255, 255]}},
    'roi': {'top_frac': 0.8, 'bottom_frac': 1.0, 'left_frac': 0.0, 'right_frac': 1.0},
    'steering': {'deadband_frac': 0.15, 'center_frac': 0.5},
    'drive': {'base_speed': 50, 'trim_left': 1.0, 'trim_right': 1.0},
    'imu': {'kind': 'angle', 'yaw_axis': 0, 'yaw_sign': 1, 'kp_heading': 1.0, 'deadband_deg': 0.5},
}


def _deep_update(dst, src):
    for k, v in src.items():
        if isinstance(v, dict) and isinstance(dst.get(k), dict):
            _deep_update(dst[k], v)
        else:
            dst[k] = v
    return dst


def load_calib():
    c = copy.deepcopy(DEFAULT_CALIB)
    try:
        with open(CALIB_PATH, 'r', encoding='utf-8') as f:
            saved = json.load(f)
        if isinstance(saved, dict):
            _deep_update(c, saved)
    except (OSError, ValueError):
        print('경고: %s 를 읽을 수 없어 기본값을 사용합니다.' % CALIB_PATH)
    return c


def _num(v):
    while isinstance(v, (list, tuple)) and len(v) > 0:
        v = v[0]
    return float(v)


CALIB = load_calib()

# 검은선은 LINE_*, 초록은 GREEN_HSV_*; 카메라/트림/IMU/엔코더는 기존 설정 사용
CAM_W, CAM_H = int(CALIB['camera']['size'][0]), int(CALIB['camera']['size'][1])
BLACK_LOWER = np.array(LINE_BLACK_LOWER, np.uint8)
BLACK_UPPER = np.array(LINE_BLACK_UPPER, np.uint8)
GREEN_LOWER = np.array(GREEN_HSV_LOWER, np.uint8)
GREEN_UPPER = np.array(GREEN_HSV_UPPER, np.uint8)
ROI = CALIB['roi']
BASE = float(BASE_SPEED_OVERRIDE if BASE_SPEED_OVERRIDE is not None else CALIB['drive']['base_speed'])
TRIM_L = float(CALIB['drive']['trim_left'])
TRIM_R = float(CALIB['drive']['trim_right'])
IMU_AXIS = int(CALIB['imu']['yaw_axis'])
IMU_SIGN = int(YAW_SIGN_OVERRIDE if YAW_SIGN_OVERRIDE is not None else CALIB['imu']['yaw_sign'])
KP_HEADING = float(CALIB['imu'].get('kp_heading', 1.0))
IMU_DEADBAND = float(YAW_DEADBAND_DEG)

TICKS_PER_REV = float(CALIB['drive'].get('ticks_per_rev', 1250))
WHEEL_RADIUS_M = float(CALIB['drive'].get('wheel_radius_m', 0.035))
_saved_mpt = float(CALIB['drive'].get('meters_per_tick', 0.0))
if METERS_PER_TICK_OVERRIDE is not None:
    METERS_PER_TICK = float(METERS_PER_TICK_OVERRIDE)
elif _saved_mpt > 0:
    METERS_PER_TICK = _saved_mpt
else:
    METERS_PER_TICK = 2.0 * math.pi * WHEEL_RADIUS_M / TICKS_PER_REV
METERS_PER_TICK *= float(DISTANCE_SCALE)
if not math.isfinite(METERS_PER_TICK) or METERS_PER_TICK <= 0:
    raise ValueError('METERS_PER_TICK이 올바르지 않습니다.')


tiki = TikiMini()
tiki.set_motor_mode(tiki.MOTOR_MODE_PID)

try:
    picam2
except NameError:
    try:
        picam2 = Picamera2()
        _cfg = picam2.create_preview_configuration(
            main={"size": (CAM_W, CAM_H), "format": CALIB['camera'].get('format', 'BGR888')})
        picam2.configure(_cfg)
        picam2.start()
        time.sleep(0.5)
    except Exception as e:
        try:
            picam2.close()
        except Exception:
            pass
        try:
            del picam2
        except NameError:
            pass
        raise RuntimeError('카메라를 열 수 없습니다. 다른 노트북의 카메라 정지 셀 실행 또는 커널 종료 후 재시도. 원인: %s' % e) from e

# 기존 카메라 노출/AWB 잠금은 유지. 원본 테스트와 노출이 달라 V=93은 현장 확인 필요.
_cc = CALIB['camera']['controls']
if CALIB['camera'].get('locked'):
    picam2.set_controls({"AwbEnable": False,
                         "ColourGains": tuple(float(x) for x in _cc['ColourGains'])})
    picam2.set_controls({"AeEnable": False, "ExposureTime": int(_cc['ExposureTime']),
                         "AnalogueGain": float(_cc['AnalogueGain'])})
    print('노출/AWB 잠금 적용 (노출 %d us, 게인 %.2f)' % (int(_cc['ExposureTime']), float(_cc['AnalogueGain'])))
else:
    print('*** 경고: calib에 노출 잠금이 없습니다(locked=false). 00 노트북에서 잠그세요 — 색이 어긋납니다. ***')

print('라인 설정: %dx%d | 검정 V=%d | 보정=%s | IMU혼합=%s | base=%.0f | yaw_sign=%+d'
      % (CAM_W, CAM_H, int(BLACK_UPPER[2]), LINE_CONTROL_MODE, LINE_USE_IMU_CORRECTION, BASE, IMU_SIGN))
print('배터리 %.2f V' % _num(tiki.get_battery_voltage()))
print('거리 환산 %.8f m/tick | wheel=%.3fm ticks/rev=%.0f scale=%.3f' % (METERS_PER_TICK, WHEEL_RADIUS_M, TICKS_PER_REV, DISTANCE_SCALE))
print('복귀 초록 HSV %s ~ %s | 최소면적 %.0f | ArUco 구역별 %d개' % (GREEN_LOWER, GREEN_UPPER, MIN_GREEN_AREA, ARUCO_REQUIRED_COUNT))


[0:14:13.667530878] [1859]  INFO Camera camera_manager.cpp:340 libcamera v0.7.2+rpt20260817
[0:14:13.676070231] [1883]  INFO RPI pisp.cpp:725 libpisp version v1.7.0 07-08-2026 (09:22:13)
[0:14:13.683805901] [1883]  INFO IPAProxy ipa_proxy.cpp:184 Using tuning file /usr/share/libcamera/ipa/rpi/pisp/imx219.json
[0:14:13.688957588] [1883]  INFO Camera camera_manager.cpp:223 Adding camera '/base/axi/pcie@1000120000/rp1/i2c@80000/imx219@10' for pipeline handler rpi/pisp
[0:14:13.688974422] [1883]  INFO RPI pisp.cpp:1186 Registered camera /base/axi/pcie@1000120000/rp1/i2c@80000/imx219@10 to CFE device /dev/media0 and ISP device /dev/media1 using PiSP variant BCM2712_D0
[0:14:13.691724375] [1883]  WARN V4L2 v4l2_pixelformat.cpp:346 Unsupported V4L2 pixel format Nc30
[0:14:13.691744172] [1883]  WARN V4L2 v4l2_pixelformat.cpp:346 Unsupported V4L2 pixel format Nc12
[0:14:13.692154449] [1859]  INFO Camera camera.cpp:1216 configuring streams: (0) 320x180-BGR888/sRGB (1) 1920x1080-BGGR_PISP_COMP1/R

노출/AWB 잠금 적용 (노출 43818 us, 게인 2.00)
라인 설정: 320x180 | 검정 V=120 | 보정=P_IMU | IMU혼합=True | base=140 | yaw_sign=-1
배터리 11.29 V
거리 환산 0.00017593 m/tick | wheel=0.035m ticks/rev=1250 scale=1.000
복귀 초록 HSV [42 72 90] ~ [ 72 199 207] | 최소면적 60 | ArUco 구역별 4개


In [3]:
# ===== 함수 정의 =====

def wrap180(a):
    while a > 180.0:
        a -= 360.0
    while a < -180.0:
        a += 360.0
    return a


def read_yaw():
    v = tiki.get_imu()
    raw = float(v[IMU_AXIS]) if hasattr(v, '__getitem__') else float(v)
    if not math.isfinite(raw):
        raise RuntimeError('IMU 값 이상 — 안전 정지')
    return IMU_SIGN * raw   # 반시계(CCW) = + 로 정규화


def convert_to_bytes(image):
    ok, buf = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return buf.tobytes() if ok else b''


def _roi_px(w, h):
    y1 = max(0, min(int(h * ROI['top_frac']), h - 1))
    y2 = max(y1 + 1, min(int(h * ROI['bottom_frac']), h))
    x1 = max(0, min(int(w * ROI['left_frac']), w - 1))
    x2 = max(x1 + 1, min(int(w * ROI['right_frac']), w))
    return x1, y1, x2, y2


ARUCO_OK = True
try:
    _ad = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    aruco_detector = cv2.aruco.ArucoDetector(_ad, cv2.aruco.DetectorParameters())
except AttributeError:
    ARUCO_OK = False
    aruco_detector = None
    print('경고: cv2.aruco 사용 불가 — ArUco 미션 비활성')


def line_roi_box(height, width):
    """맨 위 테스트의 검은 ROI. 픽셀 오프셋/폭/중심 계산 순서를 유지한다."""
    y1, y2 = int(height * LINE_ROI_TOP_FRAC), height
    roi_w = max(int(width * LINE_ROI_WIDTH_RATIO), 1)
    x1 = (width - roi_w) // 2 + LINE_ROI_OFFSET_X
    x1 = int(np.clip(x1, 0, width - roi_w))
    return x1, y1, x1 + roi_w, y2


def line_motor_command(error_px, previous_error, base_speed, frame_width=None):
    """선 위치 보정 계산. IMU/트림/최종 출력 제한과 모터 명령은 FSM에서 수행."""
    if error_px is None:
        return dict(action='LOST', left=0.0, right=0.0, prev_error=0, turn=0, base=base_speed)
    error = float(error_px)
    if LINE_CONTROL_MODE == 'P_IMU':
        width = float(CAM_W if frame_width is None else frame_width)
        if not math.isfinite(width) or width <= 0:
            raise ValueError('라인 보정 영상 폭이 올바르지 않습니다.')
        # 현재 cx-center는 우측이 +. 원본 center-cx와 부호가 반대이므로 L에 더함.
        reference_error = error * LINE_P_REFERENCE_WIDTH / width
        turn = float(np.clip(LINE_P_KP * reference_error, -LINE_P_MAX_STEER, LINE_P_MAX_STEER))
        # 원본처럼 CENTER 표시 범위 안에서도 연속 보정. D항/조향별 감속 없음.
        action = 'FORWARD' if error == 0 else ('LEFT' if error < 0 else 'RIGHT')
        return dict(action=action, left=base_speed + turn, right=base_speed - turn,
                    prev_error=error, turn=turn, base=base_speed)
    if abs(error) <= LINE_DEAD_ZONE_PX:
        # 원본은 중앙 직진에는 MAX_SPEED(120)를 적용하지 않는다.
        return dict(action='FORWARD', left=base_speed, right=base_speed,
                    prev_error=error, turn=0, base=base_speed)
    if abs(error) > LINE_MID_ERROR_PX:
        base = int(base_speed * LINE_LARGE_SPEED_FACTOR)
        kp = LINE_KP * LINE_LARGE_KP_FACTOR
    else:
        base = int(base_speed * LINE_MID_SPEED_FACTOR)
        kp = LINE_KP
    turn = int(kp * error + LINE_KD * (error - previous_error))
    left = int(np.clip(base + turn, 0, LINE_MAX_SPEED))
    right = int(np.clip(base - turn, 0, LINE_MAX_SPEED))
    return dict(action='LEFT' if error < 0 else 'RIGHT', left=left, right=right,
                prev_error=error, turn=turn, base=base)


def get_green_roi_box(black_box):
    """첨부 원본: 검은 ROI와 같은 폭, 높이 1/3, 세로 중앙 + 오프셋."""
    x1, y1, x2, y2 = black_box
    green_h = max((y2 - y1) // 3, 1)
    gy1 = y1 + ((y2 - y1) - green_h) // 2 + GREEN_OFFSET_Y
    gy1 = int(np.clip(gy1, y1, y2 - green_h))
    return x1, gy1, x2, gy1 + green_h


def detect_return_green(frame, black_box):
    """그리기 전 원본 픽셀만 분석하여 검은선의 초록 윤곽선을 오인하지 않음."""
    x1, y1, x2, y2 = get_green_roi_box(black_box)
    hsv = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour = max(contours, key=cv2.contourArea) if contours else None
    area = float(cv2.contourArea(contour)) if contour is not None else 0.0
    pixels = int(cv2.countNonZero(mask))
    return dict(found=area >= MIN_GREEN_AREA, area=area, contour=contour,
                box=(x1, y1, x2, y2), ratio=100.0 * pixels / max(1, mask.size))


def analyze(frame):
    """검은선 설정 유지, 복귀 초록은 첨부 프로필, ArUco는 원본 전체 영상 사용."""
    h, w, _ = frame.shape
    x1, y1, x2, y2 = line_roi_box(h, w)
    roi = frame[y1:y2, x1:x2]
    black = cv2.inRange(cv2.cvtColor(roi, cv2.COLOR_BGR2HSV), BLACK_LOWER, BLACK_UPPER)
    kernel = np.ones((3, 3), np.uint8)
    black = cv2.morphologyEx(black, cv2.MORPH_OPEN, kernel)
    black = cv2.morphologyEx(black, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(black, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    roi_w = x2 - x1
    center_x = roi_w // 2
    canvas = frame.copy()
    cv2.rectangle(canvas, (x1, y1), (x2 - 1, y2 - 1), (0, 165, 255), 2)
    cv2.line(canvas, (x1 + center_x, y1), (x1 + center_x, y2 - 1), (255, 255, 0), 1)

    line_error_px = None
    line_state = 'NO_LINE'
    line_signal = 0.0
    if contours:
        big = max(contours, key=cv2.contourArea)
        if cv2.contourArea(big) >= LINE_MIN_CONTOUR_AREA:
            moments = cv2.moments(big)
            if moments['m00'] != 0:
                cx = int(moments['m10'] / moments['m00'])
                cy = int(moments['m01'] / moments['m00'])
                line_error_px = cx - center_x
                if abs(line_error_px) <= LINE_DEAD_ZONE_PX:
                    line_state = 'CENTER'
                else:
                    line_state = 'LEFT' if line_error_px < 0 else 'RIGHT'
                    line_signal = -line_error_px / max(1.0, roi_w / 2.0)
                cv2.drawContours(canvas, [big + np.array([x1, y1], dtype=big.dtype)], -1, (0, 255, 0), 2)
                cv2.circle(canvas, (x1 + cx, y1 + cy), 4, (0, 0, 255), -1)

    green_obs = detect_return_green(frame, (x1, y1, x2, y2))
    gx1, gy1, gx2, gy2 = green_obs['box']
    green_ratio = green_obs['ratio']
    cv2.rectangle(canvas, (gx1, gy1), (gx2 - 1, gy2 - 1), (0, 255, 0), 1)
    if green_obs['found']:
        offset = np.array([[[gx1, gy1]]], dtype=green_obs['contour'].dtype)
        cv2.drawContours(canvas, [green_obs['contour'] + offset], -1, (0, 255, 0), 2)

    aruco_list = []
    aruco_instances = []
    if ARUCO_OK and ENABLE_ARUCO:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        corners, ids, _rej = aruco_detector.detectMarkers(gray)
        if ids is not None:
            keep_c = []
            keep_i = []
            for c, i in zip(corners, ids.flatten()):
                i = int(i)
                pts = c.reshape(4, 2)
                side = float(np.mean([np.linalg.norm(pts[k] - pts[(k + 1) % 4]) for k in range(4)]))
                aruco_list.append((i, side))
                aruco_instances.append(dict(id=i, side=side,
                                            center=pts.mean(axis=0).tolist(), corners=pts.tolist()))
                keep_c.append(c)
                keep_i.append([i])
            if keep_c:   # 사전에서 검출한 모든 ID를 화면에 표시
                cv2.aruco.drawDetectedMarkers(canvas, keep_c, np.array(keep_i, dtype=np.int32))

    ar_txt = ' '.join('id%d:%.0fpx' % (i, s) for i, s in aruco_list)
    cv2.putText(canvas, '%s sig=%+.2f green=%.1f%% %s' % (line_state, line_signal, green_ratio, ar_txt),
                (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return {'canvas': canvas, 'line_state': line_state, 'line_signal': line_signal,
            'line_error_px': line_error_px,
            'green_ratio': green_ratio, 'green_found': green_obs['found'],
            'green_area': green_obs['area'], 'aruco': aruco_list,
            'aruco_instances': aruco_instances}


def turn_deg(delta_deg, stop_evt=None, power=None):
    """제자리 회전 (누적 적분). delta>0 = 반시계(좌). 반환 (status, turned).
    stop_evt가 set되면 즉시 멈추고 'interrupted' 반환 (정지 버튼이 회전 중에도 듣게)."""
    p = float(TURN_POWER if power is None else power)
    total = 0.0
    status = 'ok'
    try:
        for attempt in range(2):
            remain = delta_deg - total
            if attempt > 0:
                if abs(remain) <= TURN_TOL_DEG:
                    break
                p *= 0.7
            goal = abs(remain) - (TURN_STOP_EARLY_DEG if attempt == 0 else TURN_STOP_EARLY_DEG * 0.5)
            if goal <= 0:
                break
            yaw_prev = read_yaw()
            turned = 0.0
            if remain > 0:
                tiki.counter_clockwise(p)
            else:
                tiki.clockwise(p)
            t0 = time.time()
            while abs(turned) < goal:
                if stop_evt is not None and stop_evt.is_set():
                    status = 'interrupted'
                    break
                if time.time() - t0 > TURN_TIMEOUT_S:
                    status = 'timeout'
                    break
                time.sleep(0.02)
                y = read_yaw()
                turned += wrap180(y - yaw_prev)
                yaw_prev = y
                if turned * remain < 0 and abs(turned) > 15.0:
                    status = 'sign_error'
                    break
            tiki.stop()
            t_s = time.time()
            while time.time() - t_s < TURN_SETTLE_S:
                time.sleep(0.02)
                y = read_yaw()
                turned += wrap180(y - yaw_prev)
                yaw_prev = y
            total += turned
            if status != 'ok':
                break
    finally:
        tiki.stop()
    return status, total


def save_aruco_json(new_ids, all_ids):
    data = {'events': []}
    try:
        with open(ARUCO_LOG_PATH, 'r', encoding='utf-8') as f:
            loaded = json.load(f)
        if isinstance(loaded, dict) and isinstance(loaded.get('events'), list):
            data = loaded
    except (OSError, ValueError):
        if os.path.exists(ARUCO_LOG_PATH):
            try:
                os.replace(ARUCO_LOG_PATH, ARUCO_LOG_PATH + '.bak')
            except OSError:
                pass
    data['events'].append({'t': datetime.now().isoformat(timespec='seconds'),
                           'new_ids': [int(i) for i in new_ids],
                           'all_ids': sorted(int(i) for i in all_ids), 'sum': int(sum(all_ids)),
                           'zone_ids': {k: list(v) for k, v in DRIVE.get('aruco_zone_ids', {}).items()},
                           'zone_complete': dict(DRIVE.get('aruco_zone_complete', {})),
                           'zone_tracks': {k: [dict(t) for t in v['items']]
                                           for k, v in DRIVE.get('aruco_tracks', {}).items()},
                           'count_mode': 'physical_instances',
                           'id_statistics': 'new_ids/all_ids/sum count unique IDs only'})
    with open(ARUCO_LOG_PATH, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


print('함수 준비 완료 (ArUco %s)' % ('사용 가능' if ARUCO_OK else '비활성'))


# 물체 모델 연결점. 예: OBJECT_CLASSIFIER = my_model_predict
# 콜백 반환값은 "CONE"/"HELMET", 또는 (label, confidence) 형식이다.
# ----- Hailo YOLO(person/helmet) 모델 연결 (asset/best_hailo_model) -----
# 지금은 person/helmet 2클래스만 있어 헬멧 감지 여부만 판단한다.
# 나중에 safety/danger/corn 3클래스 모델로 교체할 때는 HAILO_MODEL_DIR과
# classify_with_hailo() 안의 라벨 매핑만 바꾸면 된다(다른 코드는 그대로).
HAILO_MODEL_DIR = 'asset/best_hailo_model'
HAILO_CONF_THRESH = OBJECT_MIN_CONFIDENCE

try:
    from ultralytics import YOLO as _YOLO
    _hailo_model = _YOLO(HAILO_MODEL_DIR)
    print('Hailo 모델 로드 완료: %s (classes=%s)' % (HAILO_MODEL_DIR, _hailo_model.names))
except Exception as _hailo_load_error:
    _hailo_model = None
    print('경고: Hailo 모델 로드 실패(%r) — 물체 인식 비활성' % _hailo_load_error)


def classify_with_hailo(frame):
    """정지 상태에서 한 프레임만 추론. 가장 confidence 높은 박스의 클래스명 반환."""
    if _hailo_model is None:
        return None
    results = _hailo_model.predict(frame, conf=HAILO_CONF_THRESH, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return None
    confs = boxes.conf.tolist()
    best = confs.index(max(confs))
    label = _hailo_model.names[int(boxes.cls[best])]
    return label, float(confs[best])


OBJECT_CLASSIFIER = classify_with_hailo


def classify_object(frame):
    if not callable(OBJECT_CLASSIFIER):
        return None, 0.0
    result = OBJECT_CLASSIFIER(frame)
    if isinstance(result, str):
        label, confidence = result, 1.0
    elif isinstance(result, (tuple, list)) and len(result) >= 2:
        label, confidence = result[0], float(result[1])
    elif isinstance(result, dict):
        label = result.get('label')
        confidence = float(result.get('confidence', 0.0))
    else:
        return None, 0.0
    aliases = {
        'CONE': 'CONE', 'TRAFFIC_CONE': 'CONE', '라바콘': 'CONE',
        'HELMET': 'HELMET', '안전모': 'HELMET',
    }
    label = aliases.get(str(label).strip().upper(), aliases.get(str(label).strip()))
    if label not in OBJECT_LABELS or not math.isfinite(confidence):
        return None, 0.0
    return label, confidence


def mission_expected_text(step):
    event = step['event']
    if event == 'ARUCO':
        target = step['target']
        return '%s 구역 ArUco %d개 수집(ID 무관)' % (target, ARUCO_REQUIRED_COUNT)
    if event == 'OBJECT':
        return '라바콘/안전모'
    if event == 'GREEN_RETURN':
        return '%s 복귀 초록 검출' % step['return_target']
    if event == 'DISTANCE':
        return '거리 %.2fm' % step['distance_m']
    if event == 'DISTANCE_OBJECT':
        return '거리 %.2fm → 회전 중 물체 인식' % step['distance_m']
    if event == 'DISTANCE_YOLO':
        return '거리 %.2fm → 정지 후 YOLO 인식' % step['distance_m']
    return event


def mission_distance_target(step, state):
    """고정 거리 단계 전용. 복귀는 거리로 성공 판정하지 않는다."""
    if step['event'] not in ('DISTANCE', 'DISTANCE_OBJECT', 'DISTANCE_YOLO'):
        raise ValueError('초록 복귀는 거리 목표를 사용하지 않습니다.')
    distance = float(step['distance_m'])
    if not math.isfinite(distance) or distance < 0:
        raise RuntimeError('목표 거리 이상 — 안전 정지')
    return distance


def collect_zone_aruco(state, target, detections, now=None):
    """ID+중심+크기로 실물 추적. 애매한 연결/장기 누락은 완료로 인정하지 않는다.

    detections: analyze의 aruco_instances (id, side, center, corners 사전).
    한 구역의 4장이 카메라에 함께 들어오는 배치를 전제로 한 단기 추적이며,
    같은 ID 물체의 교차/장기 가림을 복원하는 일반적인 물체 추적기는 아니다.
    """
    now = time.monotonic() if now is None else float(now)
    if not math.isfinite(now):
        raise ValueError('ArUco 추적 시간 이상')
    zone = state.setdefault('aruco_tracks', {}).setdefault(
        target, {'items': [], 'next_track': 1, 'note': ''})
    tracks = zone['items']
    zone['note'] = ''
    # 일부 오래된 트랙만 지우고 새 물체를 더하면 유령 중복이 생길 수 있다.
    # 하나라도 만료되면 현재 구역 묶음 전체를 재확인한다.
    if any(now < t['last_seen'] or now - t['last_seen'] > ARUCO_TRACK_TTL_S for t in tracks):
        tracks.clear()
        zone['note'] = '오래 놓침: 구역 전체 재확인'

    visible = []
    for detection in detections:
        if not isinstance(detection, dict):
            continue  # ID/크기만 있는 옛 관측은 위치가 없어 세지 않는다.
        try:
            marker_id = int(detection['id'])
            side = float(detection['side'])
            center = [float(v) for v in detection['center']]
        except (KeyError, ValueError, TypeError, OverflowError):
            continue
        if (not 0 <= marker_id < 50 or len(center) != 2 or
                not all(math.isfinite(v) for v in center + [side]) or
                side <= 0 or side < ARUCO_MIN_SIDE_PX):
            continue
        visible.append(dict(id=marker_id, side=side, center=center))
    visible.sort(key=lambda d: (d['center'][0], d['center'][1], d['id']))
    # 같은 위치를 한 프레임에서 여러 번 받더라도 한 장만 센다.
    unique = []
    ambiguous = False
    for d in visible:
        overlaps = [p for p in unique if math.dist(d['center'], p['center']) <
                    ARUCO_TRACK_DUPLICATE_RATIO * min(d['side'], p['side'])]
        if overlaps:
            ambiguous |= any(p['id'] != d['id'] for p in overlaps)
        else:
            unique.append(d)
    visible = unique
    ambiguous |= len(visible) > ARUCO_REQUIRED_COUNT

    # 이동/크기 게이트를 양방향으로 검사한다. ID가 달라진 동일 위치도
    # 새 물체로 추가하지 않고 불확실한 관측으로 취급한다.
    candidates = {}
    reverse = {j: [] for j in range(len(tracks))}
    for i, d in enumerate(visible):
        candidates[i] = []
        for j, t in enumerate(tracks):
            scale = max(d['side'], t['side']) / min(d['side'], t['side'])
            shift = math.dist(d['center'], t['center'])
            if (scale <= ARUCO_TRACK_MAX_SCALE and
                    shift <= max(4.0, ARUCO_TRACK_MAX_SHIFT * max(d['side'], t['side']))):
                candidates[i].append(j)
                reverse[j].append(i)
    matches = {}
    for i, js in candidates.items():
        if len(js) == 1 and len(reverse[js[0]]) == 1 and visible[i]['id'] == tracks[js[0]]['id']:
            matches[i] = js[0]
        elif js:
            ambiguous = True

    added = []
    for i, j in matches.items():
        d, t = visible[i], tracks[j]
        t.update(center=d['center'], side=d['side'], last_seen=now, hits=t['hits'] + 1)
        if not t['confirmed'] and t['hits'] >= ARUCO_CONFIRM_FRAMES:
            t['confirmed'] = True
            added.append(t['id'])

    new_observations = []
    for i, d in enumerate(visible):
        if i in matches or candidates[i]:
            continue
        # 同 ID의 기존 물체를 놓친 상태라면 멀어진 새 검출이 그 물체일 수 있다.
        # 기존 同 ID들이 모두 연결된 프레임에서만 별도 위치를 새 장으로 추가.
        if any(t['id'] == d['id'] and j not in matches.values() for j, t in enumerate(tracks)):
            ambiguous = True
            continue
        new_observations.append(d)
    if len(tracks) + len(new_observations) > ARUCO_REQUIRED_COUNT:
        ambiguous = True
        new_observations = []
    for d in new_observations:
        confirmed = ARUCO_CONFIRM_FRAMES <= 1
        tracks.append(dict(track=zone['next_track'], id=d['id'], center=d['center'],
                           side=d['side'], last_seen=now, hits=1, confirmed=confirmed))
        zone['next_track'] += 1
        if confirmed:
            added.append(d['id'])

    if ambiguous:
        # 다음 빈 프레임이 불확실 판정을 해제해 오래된 4장을 완료시키지 않게
        # 위치 트랙은 유지하되 확인 횟수를 새로 받는다.
        for t in tracks:
            t.update(hits=0, confirmed=False)
        added = []
    ids = [t['id'] for t in tracks if t['confirmed']]
    state.setdefault('aruco_zone_ids', {})[target] = ids  # 중복 ID 보존
    state.setdefault('aruco_zone_complete', {})[target] = (
        len(ids) == ARUCO_REQUIRED_COUNT and not ambiguous)
    if ambiguous:
        zone['note'] = '위치 연결 불확실: 확인 횟수 초기화, 완료 보류'
    state.setdefault('seen_ids', set()).update(added)  # ID 종류 통계만, 완료 조건 아님
    return added


def confirm_return_green(state, found):
    """초기 잔류는 clear 후 재검출. 유효 검출 첫 프레임부터 정지 확인."""
    if state.get('return_green_armed') is None:
        state['return_green_armed'] = not (GREEN_IGNORE_INITIAL_VISIBLE and found)
    if not found:
        state['return_green_armed'] = True
        state['return_green_streak'] = 0
        return 'SEEK'
    if not state['return_green_armed']:
        state['return_green_streak'] = 0
        return 'WAIT_CLEAR'
    state['return_green_streak'] = state.get('return_green_streak', 0) + 1
    return 'CONFIRMED' if state['return_green_streak'] >= GREEN_CONFIRM_FRAMES else 'CANDIDATE'


def read_encoder_pair():
    left = _num(tiki.get_encoder(tiki.MOTOR_LEFT))
    right = _num(tiki.get_encoder(tiki.MOTOR_RIGHT))
    if not (math.isfinite(left) and math.isfinite(right)):
        raise RuntimeError('엔코더 값 이상 — 안전 정지')
    return left, right


def encoder_distance_m(start_pair):
    left, right = read_encoder_pair()
    start_left, start_right = start_pair
    avg_ticks = (abs(left - start_left) + abs(right - start_right)) / 2.0
    return avg_ticks * METERS_PER_TICK


def turn_deg_with_recognition(delta_deg, stop_evt=None, power=None):
    """회전 제어와 카메라 추론을 병렬 실행하고 다중 프레임 투표 결과를 반환한다."""
    recog_stop = threading.Event()
    samples = []

    def _recognition_worker():
        while not recog_stop.is_set() and not (stop_evt is not None and stop_evt.is_set()):
            try:
                manual = globals().get('DRIVE', {}).get('manual_object')
                if manual in OBJECT_LABELS:
                    globals()['DRIVE']['manual_object'] = None
                    samples.append((manual, 1.0))
                else:
                    frame = picam2.capture_array()
                    if CALIB['camera'].get('rgb2bgr', True):
                        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
                    label, confidence = classify_object(frame)
                    if label in OBJECT_LABELS and confidence >= OBJECT_MIN_CONFIDENCE:
                        samples.append((label, float(confidence)))
            except Exception as exc:
                # 회전 제어는 계속 유지하되 마지막 인식 오류를 결과에 남긴다.
                globals().get('DRIVE', {})['last_object_error'] = repr(exc)
            time.sleep(OBJECT_SAMPLE_INTERVAL_S)

    worker = threading.Thread(target=_recognition_worker, daemon=True)
    worker.start()
    try:
        status, turned = turn_deg(delta_deg, stop_evt, power)
    finally:
        recog_stop.set()
        worker.join(timeout=1.0)

    scores = {label: 0.0 for label in OBJECT_LABELS}
    counts = {label: 0 for label in OBJECT_LABELS}
    for label, confidence in samples:
        scores[label] += confidence
        counts[label] += 1
    if not samples:
        return status, turned, {'label': 'UNKNOWN', 'confidence': 0.0, 'samples': 0}
    label = max(OBJECT_LABELS, key=lambda x: (scores[x], counts[x]))
    confidence = scores[label] / max(1, counts[label])
    return status, turned, {
        'label': label,
        'confidence': confidence,
        'samples': counts[label],
    }

함수 준비 완료 (ArUco 사용 가능)


## <b>턴 테스트 (권장)</b>

주행 전 회전 실측. **잔차 + (부족)** → `TURN_STOP_EARLY_DEG` 줄이기, **− (과회전)** → 늘리기 후 상수 셀 재실행.

In [4]:
# 턴 테스트
_out_tt = widgets.Output()


def _turn_test(deg):
    with _out_tt:
        try:
            print('배터리 %.2f V | %+.0f도... (yaw_sign=%+d)' % (_num(tiki.get_battery_voltage()), deg, IMU_SIGN))
            status, turned = turn_deg(deg, None)
            print('결과 %s | 실제 %+.1f도 (잔차 %+.1f도)' % (status, turned, deg - turned))
            if status == 'sign_error':
                print('  ★ 명령과 반대로 측정됨 -> 셀 1의 YAW_SIGN_OVERRIDE를 %+d 로 바꾸고' % (-IMU_SIGN))
                print('    셀 1~3을 다시 실행한 뒤 재시도하세요.')
            elif status == 'ok':
                print('  OK. 로봇이 실제로 %s 방향으로 돌았는지 눈으로 확인하세요' % ('왼쪽(반시계)' if deg > 0 else '오른쪽(시계)'))
                print('  (반대로 돌면 회전 방향 상수의 부호 문제 — 알려주세요).')
            print()
        finally:
            tiki.stop()


_b1 = widgets.Button(description='좌 90°', button_style='info')
_b2 = widgets.Button(description='우 90°', button_style='info')
_b3 = widgets.Button(description='180°', button_style='warning')
_b1.on_click(lambda b: _turn_test(+90.0))
_b2.on_click(lambda b: _turn_test(-90.0))
_b3.on_click(lambda b: _turn_test(+180.0))
display(widgets.HBox([_b1, _b2, _b3]), _out_tt)

Output()

## 테스트 경기장 혼합 FSM 주행

`DISTANCE` / `DISTANCE_OBJECT`: 고정 거리로 진입·순찰·최종 정지.
`ARUCO`: 위치로 구분한 실물 마커 4장을 확인하면 정지·180도 (같은 ID 허용).
`GREEN_RETURN`: 검은선을 따라 저속 복귀하며 초록 검출 후 정지·좌90.

영상 아래 이벤트 문구에서 구역별 ID 리스트와 `SEEK / WAIT_CLEAR / CANDIDATE` 상태를 확인하세요.
순찰 #1 → #2 및 #3 → 정지구역 사이의 직진은 유지합니다.


In [5]:
# ===== 거리 기반 10상태 FSM 주행 =====
video_widget = widgets.Image(
    format='jpeg',
    layout=widgets.Layout(width='%dpx' % (CAM_W * 2), height='%dpx' % (CAM_H * 2)),
)
status_widget = widgets.Label(value='대기 중 — 거리 파라미터를 확인하세요')
event_widget = widgets.Label(value='')
btn_start = widgets.Button(description='주행 시작', button_style='success')
btn_stop = widgets.Button(description='정지', button_style='danger')
btn_reset = widgets.Button(description='기록 초기화', button_style='warning')
btn_cone = widgets.Button(description='수동: 라바콘 확인', button_style='info')
btn_helmet = widgets.Button(description='수동: 안전모 확인', button_style='info')

DRIVE = {'seen_ids': set(), 'mission_step': 0, 'manual_object': None}


def beep(freq, dur=0.15):
    try:
        tiki.play_buzzer(freq)
        time.sleep(dur)
    finally:
        try:
            tiki.stop_buzzer()
        except Exception:
            pass


def _drive_alive():
    t = globals().get('_drive_thread')
    return t is not None and t.is_alive()


def _drive_loop(stop_evt):
    S = DRIVE
    S.update({
        'mission_step': 0,
        'aruco_after': 0.0,
        'seen_ids': set(),
        'aruco_zone_ids': {},
        'aruco_tracks': {},
        'aruco_zone_complete': {},
        'return_green_armed': None,
        'return_green_streak': 0,
        'object_label': None,
        'object_streak': 0,
        'object_gate_open': False,
        'manual_object': None,
        'line_prev_error': 0.0,
        'finished': False,
        'patrol_results': [],
        'last_object_error': None,
    })
    pending_ids = set()  # 주행 중 파일 쓰기 대신, 완료/종료 시 정지 후 저장
    last_ar_log_t = 0.0
    line_preview_count = 0
    start_yaw = read_yaw()
    segment_origin = read_encoder_pair()
    segment_started_at = time.time()
    start_time = time.time()

    def reset_detectors():
        S['line_prev_error'] = 0.0  # 다른 구간/회전 전 오차로 D항이 튀지 않게 초기화
        S['return_green_armed'] = None
        S['return_green_streak'] = 0
        S['object_label'] = None
        S['object_streak'] = 0
        S['object_gate_open'] = False
        S['manual_object'] = None

    def reset_segment():
        nonlocal segment_origin, segment_started_at
        segment_origin = read_encoder_pair()
        segment_started_at = time.time()

    def scan_guard(step, distance=None):
        """전진 상한은 초록/수집 성공 조건이 아니라 실패 시 정지 조건."""
        if step['event'] not in ('ARUCO', 'GREEN_RETURN'):
            return
        target = step.get('target', step.get('return_target'))
        collecting = step['event'] == 'ARUCO'
        limit_m = (ARUCO_MAX_APPROACH_M if collecting else GREEN_RETURN_MAX_M)[target]
        limit_s = ARUCO_SCAN_TIMEOUT_S if collecting else GREEN_RETURN_TIMEOUT_S
        distance = encoder_distance_m(segment_origin) if distance is None else distance
        if distance >= limit_m or time.time() - segment_started_at >= limit_s:
            tiki.stop()
            raise RuntimeError('%s %s 한도 도달: %.3f/%.3fm, %.1f/%.1fs — 안전 정지' % (
                target, 'ArUco 수집' if collecting else '초록 복귀', distance, limit_m,
                time.time() - segment_started_at, limit_s))

    def do_turn_and_reset(deg, why, recognize=False):
        nonlocal start_yaw
        tiki.stop()
        t0 = time.time()
        while time.time() - t0 < TURN_PRESTOP_S:
            if stop_evt.is_set():
                return 'interrupted', 0.0, None
            tiki.stop()
            time.sleep(0.05)
        if stop_evt.is_set() or time.time() - start_time >= MAX_RUN_TIME:
            return 'interrupted', 0.0, None
        if recognize:
            status, turned, recognition = turn_deg_with_recognition(deg, stop_evt)
        else:
            status, turned = turn_deg(deg, stop_evt)
            recognition = None
        if status == 'ok':
            start_yaw = wrap180(start_yaw + deg)
            S['aruco_after'] = time.time() + ARUCO_COOLDOWN_S
        event_widget.value = '%s → %+.0f도 (%s, 실제 %+.1f도)' % (why, deg, status, turned)
        return status, turned, recognition

    def advance_step(detail):
        completed = MISSION_PLAN[S['mission_step']]['name']
        S['mission_step'] += 1
        reset_detectors()
        reset_segment()
        if S['mission_step'] >= len(MISSION_PLAN):
            S['finished'] = True
            event_widget.value = '%s 완료 | %s' % (completed, detail)
        else:
            nxt = MISSION_PLAN[S['mission_step']]
            event_widget.value = '%s 완료 | 다음 %d/13: %s' % (
                completed, S['mission_step'] + 1, mission_expected_text(nxt))

    def execute_step(step, detected_detail):
        tiki.stop()
        if step['action'] == 'FINISH':
            beep(1047, 0.12)
            beep(1319, 0.18)
            advance_step(detected_detail)
            return 'finished'
        beep(880 if step['event'] == 'ARUCO' else 660)
        if step['event'] == 'DISTANCE_YOLO':
            # 회전 없이 정지 상태에서 한 프레임만 추론 (스레드 불필요).
            frame = picam2.capture_array()
            if CALIB['camera'].get('rgb2bgr', True):
                frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            label, confidence = classify_object(frame)
            object_detail = '%s conf=%.2f' % (label or 'UNKNOWN', confidence)
            detected_detail += ' | ' + object_detail
            S['patrol_results'].append({
                'state': S['mission_step'] + 1,
                'name': step['name'],
                'result': {'label': label or 'UNKNOWN', 'confidence': confidence,
                           'samples': 1 if label else 0},
            })
            tiki.log('OBJ ' + object_detail)
        degrees = {'LEFT_90': 90.0, 'RIGHT_270': -270.0, 'TURN_180': 180.0, 'NONE': 0.0}[step['action']]
        patrol_scan = step['event'] == 'DISTANCE_OBJECT'
        status, _turned, recognition = do_turn_and_reset(
            degrees, step['name'] + ' | ' + detected_detail, recognize=patrol_scan)
        if patrol_scan and recognition is not None:
            object_detail = '%s conf=%.2f samples=%d' % (
                recognition['label'], recognition['confidence'], recognition['samples'])
            detected_detail += ' | ' + object_detail
            S['patrol_results'].append({
                'state': S['mission_step'] + 1,
                'name': step['name'],
                'result': dict(recognition),
            })
            tiki.log('OBJ ' + object_detail)
        if status == 'ok':
            advance_step(detected_detail)
        return status

    try:
        while not stop_evt.is_set() and time.time() - start_time < MAX_RUN_TIME:
            now = time.time()
            if S['mission_step'] >= len(MISSION_PLAN) or S['finished']:
                break
            step = MISSION_PLAN[S['mission_step']]
            traveled_m = encoder_distance_m(segment_origin)
            scan_guard(step, traveled_m)

            # 거리 이벤트는 영상 분석보다 먼저 검사한다.
            if step['event'] in ('DISTANCE', 'DISTANCE_OBJECT', 'DISTANCE_YOLO'):
                target_m = mission_distance_target(step, S)
                trigger_m = max(0.0, target_m - DISTANCE_STOP_EARLY_M)
                if not S['object_gate_open'] and traveled_m >= trigger_m:
                    detail = '거리 %.3f/%.3fm' % (traveled_m, target_m)
                    # 순찰구역은 거리 도달 즉시 회전하며 그동안 물체를 인식한다.
                    result = execute_step(step, detail)
                    if result == 'finished':
                        status_widget.value = '13/13 정지구역 도착 — 미션 완료'
                        break
                    if result == 'interrupted':
                        break
                    if result != 'ok':
                        status_widget.value = '회전 이상(%s) — 현재 상태 유지, 안전 정지' % result
                        break
                    continue
                if (not S['object_gate_open'] and
                        now - segment_started_at > DISTANCE_SEGMENT_TIMEOUT_S):
                    status_widget.value = '거리 단계 타임아웃 — 엔코더/거리값 확인, 안전 정지'
                    break

            yaw = read_yaw()
            yaw_error = wrap180(yaw - start_yaw)
            if abs(yaw_error) < IMU_DEADBAND:
                yaw_correction = 0.0
            else:
                yaw_correction = max(
                    min(YAW_KP * yaw_error, YAW_MAX_CORRECTION), -YAW_MAX_CORRECTION)

            frame = picam2.capture_array()
            if CALIB['camera'].get('rgb2bgr', True):
                frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            obs = analyze(frame)

            # 영상 수신/분석이 늦었을 때도 정지 요청과 제한시간이 새 동작보다 우선한다.
            if stop_evt.is_set() or time.time() - start_time >= MAX_RUN_TIME:
                break
            # 캡처/분석 지연 후 최신 엔코더·시간 한도를 다시 확인한다.
            scan_guard(step)
            triggered = False
            detected_detail = ''
            green_phase = None
            if step['event'] == 'ARUCO':
                target = step['target']
                added = (collect_zone_aruco(S, target, obs['aruco_instances'], now=time.monotonic())
                         if time.time() >= S['aruco_after'] else [])
                ids = S['aruco_zone_ids'].setdefault(target, [])
                pending_ids.update(added)
                if S['aruco_zone_complete'].get(target, False):
                    # 네 번째 확정 즉시 정지: OLED/파일/화면 갱신보다 우선.
                    tiki.stop()
                    triggered = True
                    detected_detail = '%s IDs=%s (%d/%d)' % (target, ids, len(ids), ARUCO_REQUIRED_COUNT)
                tracking = S['aruco_tracks'].get(target, {'items': [], 'note': ''})
                track_text = ' '.join('T%d:id%d=%d%s' % (
                    t['track'], t['id'], t['hits'], '*' if t['confirmed'] else '')
                    for t in tracking['items'])
                event_widget.value = '%s IDs=%s (%d/%d장) | %s | %s' % (
                    target, ids, len(ids), ARUCO_REQUIRED_COUNT, track_text, tracking['note'])
            elif step['event'] == 'GREEN_RETURN':
                green_phase = confirm_return_green(S, obs['green_found'])
                if green_phase in ('CANDIDATE', 'CONFIRMED'):
                    tiki.stop()  # 첫 유효 초록부터 화면 출력보다 먼저 정지
                event_widget.value = '%s 초록 %s | area=%.0f | 확인 %d/%d' % (
                    step['return_target'], green_phase, obs['green_area'],
                    S['return_green_streak'], GREEN_CONFIRM_FRAMES)
                if green_phase == 'CONFIRMED':
                    triggered = True
                    detected_detail = '%s 복귀 초록 area=%.0f' % (step['return_target'], obs['green_area'])

            if ARUCO_OLED_LOG and obs['aruco'] and now - last_ar_log_t > 1.0:
                tiki.log('see ' + ' '.join('id%d(%.0f)' % (i, side) for i, side in obs['aruco'][:4]))
                last_ar_log_t = now
            if stop_evt.is_set() or time.time() - start_time >= MAX_RUN_TIME:
                break
            scan_guard(step)
            if triggered:
                if step['event'] == 'ARUCO':
                    save_aruco_json(sorted(pending_ids), S['seen_ids'])
                    pending_ids.clear()
                    tiki.log(detected_detail)
                if stop_evt.is_set() or time.time() - start_time >= MAX_RUN_TIME:
                    break
                scan_guard(step)
                result = execute_step(step, detected_detail)
                if result == 'interrupted':
                    break
                if result != 'ok':
                    status_widget.value = '회전 이상(%s) — %d단계 유지, 안전 정지' % (
                        result, S['mission_step'] + 1)
                    break
                continue
            if green_phase == 'CANDIDATE':
                status_widget.value = '%d/13 복귀 초록 정지 확인 %d/%d' % (
                    S['mission_step'] + 1, S['return_green_streak'], GREEN_CONFIRM_FRAMES)
                video_widget.value = convert_to_bytes(obs['canvas'])
                time.sleep(LOOP_DT)
                continue

            drive_base = BASE
            remaining_m = None
            if step['event'] == 'ARUCO':
                drive_base = min(float(BASE), float(ARUCO_APPROACH_SPEED))
            elif step['event'] == 'GREEN_RETURN':
                drive_base = min(float(BASE), float(GREEN_RETURN_SPEED))
            elif step['event'] in ('DISTANCE', 'DISTANCE_OBJECT', 'DISTANCE_YOLO'):
                remaining_m = max(0.0, target_m - traveled_m)
                if remaining_m <= DISTANCE_SLOWDOWN_M:
                    drive_base *= DISTANCE_SLOW_FACTOR

            command = line_motor_command(
                obs['line_error_px'], S['line_prev_error'], drive_base, frame_width=frame.shape[1])
            S['line_prev_error'] = command['prev_error']
            steer = command['turn']  # 표시용 선 보정 출력: +는 오른쪽 조향
            if stop_evt.is_set() or time.time() - start_time >= MAX_RUN_TIME:
                break
            if command['action'] == 'LOST':
                tiki.stop()
                left_power = right_power = 0.0
            else:
                if LINE_USE_IMU_CORRECTION and obs['line_state'] == 'CENTER' and HEADING_RESYNC > 0:
                    start_yaw = wrap180(start_yaw + HEADING_RESYNC * wrap180(yaw - start_yaw))
                correction = yaw_correction if LINE_USE_IMU_CORRECTION else 0.0
                # read_yaw는 CCW=+로 정규화됨. +오차는 좌측 출력↑/우측↓로 복원.
                power_limit = (float(LINE_MAX_SPEED)
                               if LINE_CONTROL_MODE == 'PD_TEST' and command['action'] != 'FORWARD'
                               else 180.0)
                left_power = min(power_limit, max(0.0, command['left'] * TRIM_L + correction))
                right_power = min(power_limit, max(0.0, command['right'] * TRIM_R - correction))
                tiki.set_motor_power(tiki.MOTOR_LEFT, left_power)
                tiki.set_motor_power(tiki.MOTOR_RIGHT, right_power)

            expected = mission_expected_text(step)
            distance_txt = '%.3fm' % traveled_m
            if remaining_m is not None:
                distance_txt += ' (남음 %.3fm)' % remaining_m
            object_txt = '-'
            if step['event'] == 'OBJECT':
                object_txt = '%s %d/%d' % (
                    S['object_label'] or '대기', S['object_streak'], OBJECT_CONFIRM_FRAMES)
            # 인식/제어는 매 프레임, JPEG/위젯 갱신만 원본처럼 2프레임마다 수행.
            line_preview_count += 1
            if line_preview_count % LINE_DISPLAY_EVERY == 0:
                error_txt = '-' if obs['line_error_px'] is None else '%+dpx' % obs['line_error_px']
                status_widget.value = (
                    '%d/13 %s | 기대=%s | 거리=%s | yaw=%+.1f | %s error=%s %s=%+.1f | '
                    'object=%s | L/R=%.1f/%.1f' % (
                        S['mission_step'] + 1, step['name'], expected, distance_txt, yaw_error,
                        obs['line_state'], error_txt, LINE_CONTROL_MODE, steer, object_txt, left_power, right_power))
                video_widget.value = convert_to_bytes(obs['canvas'])
            time.sleep(LOOP_DT)

        if S['finished']:
            status_widget.value = '13/13 정지구역 도착 — 미션 완료'
        elif stop_evt.is_set():
            status_widget.value = '정지됨 (사용자) | 현재 %d/13' % min(S['mission_step'] + 1, 13)
        elif time.time() - start_time >= MAX_RUN_TIME:
            status_widget.value = '제한시간 종료 | 현재 %d/13' % min(S['mission_step'] + 1, 13)
    except Exception as e:
        status_widget.value = '오류로 정지: %r | 현재 %d/13' % (
            e, min(S.get('mission_step', 0) + 1, 13))
    finally:
        tiki.stop()
        if pending_ids:
            try:
                save_aruco_json(sorted(pending_ids), S['seen_ids'])
            except Exception as save_error:
                event_widget.value = '정지 완료, 미완료 수집 기록 저장 실패: %r' % save_error
        tiki.log('END state=%d/13 sum=%d' % (
            min(S.get('mission_step', 0), 13), int(sum(S['seen_ids']))))


def _on_start(_b=None):
    global _drive_thread, _drive_stop
    if _drive_alive():
        event_widget.value = '이미 주행 중입니다.'
        return
    if not DISTANCES_CONFIGURED:
        event_widget.value = '거리값 미확정: 셀 1 측정값 입력 후 DISTANCES_CONFIGURED=True로 변경하세요.'
        return
    if not ARUCO_OK:
        event_widget.value = 'ArUco 기능을 사용할 수 없어 FSM을 시작할 수 없습니다.'
        return
    v = _num(tiki.get_battery_voltage())
    if v < LOW_BATT_V:
        event_widget.value = '경고: 배터리 %.2f V — 충전 권장 (그래도 시작함)' % v
    DRIVE['seen_ids'] = set()
    DRIVE['mission_step'] = 0
    DRIVE['manual_object'] = None
    _drive_stop = make_stop_event()
    _drive_thread = threading.Thread(target=_drive_loop, args=(_drive_stop,), daemon=True)
    _drive_thread.start()
    status_widget.value = '1/13 연습경기장 주행 시작...'


def _on_stop(_b=None):
    try:
        _drive_stop.set()
    except NameError:
        pass
    tiki.stop()
    status_widget.value = '정지 요청 — 멈추는 중...'


def _manual_object(label):
    if not _drive_alive():
        event_widget.value = '주행 중이 아닙니다.'
        return
    idx = DRIVE.get('mission_step', 0)
    if idx >= len(MISSION_PLAN) or MISSION_PLAN[idx]['event'] != 'DISTANCE_OBJECT':
        event_widget.value = '현재 단계는 순찰 물체 단계가 아닙니다.'
        return
    DRIVE['manual_object'] = label
    event_widget.value = '%s 수동 인식 예약 — 순찰 회전에서 사용' % label


def _on_reset(_b=None):
    if _drive_alive():
        event_widget.value = '주행 중에는 초기화할 수 없습니다. [정지] 후 눌러주세요.'
        return
    DRIVE['seen_ids'] = set()
    DRIVE['mission_step'] = 0
    DRIVE['manual_object'] = None
    DRIVE['aruco_zone_ids'] = {}
    DRIVE['aruco_tracks'] = {}
    DRIVE['aruco_zone_complete'] = {}
    DRIVE['return_green_armed'] = None
    DRIVE['return_green_streak'] = 0
    if os.path.exists(ARUCO_LOG_PATH):
        try:
            os.replace(ARUCO_LOG_PATH, ARUCO_LOG_PATH + '.old')
        except OSError:
            pass
    event_widget.value = 'FSM/ArUco 기록 초기화 — 1단계 준비'


btn_start.on_click(_on_start)
btn_stop.on_click(_on_stop)
btn_reset.on_click(_on_reset)
btn_cone.on_click(lambda _b: _manual_object('CONE'))
btn_helmet.on_click(lambda _b: _manual_object('HELMET'))
display(
    widgets.HBox([btn_start, btn_stop, btn_reset]),
    widgets.HBox([btn_cone, btn_helmet]),
    video_widget,
    status_widget,
    event_widget,
)

Image(value=b'', format='jpeg', layout="Layout(height='360px', width='640px')")

Label(value='대기 중 — 거리 파라미터를 확인하세요')

Label(value='')

## <b>정리</b>

세션 종료 시 실행 — 모터 정지 + 카메라 해제 (다른 노트북에서 카메라 쓰려면 필수).

In [ ]:
# 정리 — 스레드 정지 -> 모터 정지 -> 카메라 해제
for _e in _stop_events:
    _e.set()
try:
    _drive_thread.join(timeout=3.0)
except NameError:
    pass
try:
    tiki.stop()
    tiki.stop_buzzer()
    print('모터 정지 완료')
except Exception as _e:
    print('모터 정지 오류:', _e)

if 'picam2' in globals():
    for _fn in ('stop', 'close'):
        try:
            getattr(picam2, _fn)()
        except Exception as _e:
            print('카메라 %s 오류: %s' % (_fn, _e))
    del picam2
    print('카메라 해제 완료')
else:
    print('카메라가 이미 해제되어 있습니다.')